# 面试问题：LLM-as-a-Judge 能直接当评测真值吗，怎样校准偏差？

**一句话回答**：不能。Judge 是有位置、长度、风格、自我偏好和随机性的测量工具。先写原子 rubric 与结构化输出，用人工标注集测准确率/一致性；pairwise 评测交换 A/B 顺序，冲突则 tie/复核；控制答案长度与身份泄漏，重复采样估稳定性，并把 judge 模型、prompt 和解析器完整版本化。

本 Notebook 用受控“模拟 Judge”显式制造位置/冗长偏差，再实现 swap、校准、人工一致性和多 Judge 聚合。

In [ ]:
from dataclasses import dataclass
from collections import Counter
import hashlib, json, math
import numpy as np

SEED105=10501; rng105=np.random.default_rng(SEED105)
assert SEED105==10501
assert Counter("AAB")["A"]==2
assert hashlib.sha256(b"judge-v1").hexdigest()!=hashlib.sha256(b"judge-v2").hexdigest()

## 1. Rubric 必须原子、可观察、有拒绝非法输出的 parser

不要只问“哪个更好”。把正确性、证据、完整性和安全分别定义，说明 tie 条件；输出固定为 winner、各维分数和理由引用。parser 对未知键、越界分数和自由文本失败关闭，不能把解析失败默认为通过。

In [ ]:
REQUIRED105={"winner","correctness","groundedness","reason"}
def parse_judgment105(obj):
    if set(obj)!=REQUIRED105 or obj["winner"] not in {"A","B","tie"}: raise ValueError("judge_schema")
    if not all(isinstance(obj[k],int) and 1<=obj[k]<=5 for k in ("correctness","groundedness")) or not isinstance(obj["reason"],str): raise ValueError("judge_schema")
    return obj
valid105=parse_judgment105({"winner":"A","correctness":5,"groundedness":4,"reason":"命中证据"})
assert valid105["winner"]=="A" and valid105["correctness"]==5
try: parse_judgment105({"winner":"A","correctness":9,"groundedness":4,"reason":"x"}); raise AssertionError("bad score accepted")
except ValueError as e: assert str(e)=="judge_schema"
assert REQUIRED105==set(valid105)

## 2. 位置偏差的受控反例

下面把真实质量差与“偏爱 A 位置”的偏置相加。质量接近时，Judge 会系统性选 A；差距大时偏置较难翻转。受控合成实验不是在估计某个真实模型，而是验证评测管线能否发现已知偏差。

In [ ]:
quality_a105=rng105.normal(0,1,400); quality_b105=quality_a105+rng105.normal(0,.15,400); truth105=np.where(quality_a105>quality_b105,"A","B")
def biased_pair105(qa,qb,a_len,b_len,position_bias=.12,verbosity_bias=.002):
    margin=qa-qb+position_bias+verbosity_bias*(a_len-b_len); return "A" if margin>.02 else "B" if margin<-.02 else "tie"
raw105=np.array([biased_pair105(a,b,100,100) for a,b in zip(quality_a105,quality_b105)])
assert np.mean(raw105=="A")>.7
assert .4<np.mean(truth105=="A")<.6
assert np.mean(raw105==truth105)<.75

## 3. A/B swap 能检测而非神奇消除偏差

同一 pair 评两次：原顺序与交换顺序。把交换结果映射回原身份；两次一致才接受，否则记 tie 或送人工。该策略牺牲覆盖率与两倍成本换取更高可信度，不能消除双方答案共享的风格偏差。

In [ ]:
def swap_consensus105(qa,qb,la,lb):
    first=biased_pair105(qa,qb,la,lb); swapped=biased_pair105(qb,qa,lb,la); mapped={"A":"B","B":"A","tie":"tie"}[swapped]
    return first if first==mapped else "tie"
consensus105=np.array([swap_consensus105(a,b,100,100) for a,b in zip(quality_a105,quality_b105)])
decided105=consensus105!="tie"
assert decided105.mean()<.5
assert np.mean(consensus105[decided105]==truth105[decided105])>.9
assert set(consensus105)=={"A","B","tie"}

## 4. 冗长/格式偏差要用 counterfactual probe 测

保持事实内容不变，只扩写措辞或交换格式，理想 Judge 应给 tie。若偏好长答案，可在 rubric 强调“无关内容不加分”、限制长度、同时提供原子事实表，或用人工校准拟合偏置项。不要直接截断到丢失证据。

In [ ]:
concise_len105,long_len105=40,240; same_quality105=.8
prefer_long105=biased_pair105(same_quality105,same_quality105,concise_len105,long_len105,position_bias=0,verbosity_bias=.003)
def calibrated_pair105(qa,qb,la,lb,verbosity=.002): return biased_pair105(qa,qb,la,lb,position_bias=0,verbosity_bias=0)
calibrated105=calibrated_pair105(same_quality105,same_quality105,concise_len105,long_len105)
assert prefer_long105=="B"
assert calibrated105=="tie"
assert long_len105/concise_len105==6

## 5. 用盲化人工集校准 Judge

人工 gold 也不是绝对真理，应有清晰指南、双标和仲裁。报告 Judge 对人工的准确率、混淆矩阵以及 Cohen's κ；高总体准确率可能掩盖 Judge 从不判 tie。模型身份、供应商和答案位置要盲化。

In [ ]:
human105=np.array(["A","A","B","tie","B","A","tie","B"]); model105=np.array(["A","B","B","tie","B","A","A","B"]); labels105=["A","B","tie"]
def cohen_kappa105(a,b,labels):
    po=np.mean(a==b); pa=np.array([np.mean(a==x) for x in labels]); pb=np.array([np.mean(b==x) for x in labels]); pe=float(pa@pb); return (po-pe)/(1-pe)
acc105=float(np.mean(human105==model105)); kappa105=cohen_kappa105(human105,model105,labels105)
assert math.isclose(acc105,.75)
assert 0<kappa105<1
assert sum(np.sum((human105==x)&(model105==y)) for x in labels105 for y in labels105)==len(human105)

## 6. 重复采样区分随机噪声和系统偏差

对边界 case 用多个 seed/temperature 重复评判，报告一致率与 winner 分布。高重复一致性只说明稳定，不说明正确；若稳定地偏爱 A，反而是系统偏差证据。生产回归尽量固定模型快照和采样设置。

In [ ]:
def noisy_votes105(margin,n=101,seed=0):
    rg=np.random.default_rng(seed); noisy=margin+rg.normal(0,.08,n); return np.where(noisy>.02,"A",np.where(noisy<-.02,"B","tie"))
votes105=noisy_votes105(.03,501,105); counts105=Counter(votes105); agreement105=max(counts105.values())/len(votes105)
assert sum(counts105.values())==501
assert set(counts105)<= {"A","B","tie"}
assert .3<agreement105<.8

## 7. 多 Judge 聚合不是简单多数票万能药

多模型只有在错误不完全相关时才有收益。先测每个 Judge 与人工的一致性及 Judge 间相关，再按校准可靠性加权；高风险或分歧样本交人工。下面实现带 abstain margin 的加权投票。

In [ ]:
def weighted_vote105(votes,weights,margin=.15):
    score=Counter()
    for v,w in zip(votes,weights):
        if v!="tie": score[v]+=w
    total=sum(score.values())
    if total==0 or abs(score["A"]-score["B"])/total<margin: return "tie"
    return "A" if score["A"]>score["B"] else "B"
assert weighted_vote105(["A","A","B"],[.8,.7,.9])=="A"
assert weighted_vote105(["A","B"],[.8,.8])=="tie"
assert weighted_vote105(["tie","tie"],[1,1])=="tie"

## 8. Judge 版本升级也必须走评测与灰度

Judge model、system prompt、rubric、few-shot、采样参数和 parser 任一变化都会改变测量尺度。先在固定人工集重标定，再 shadow 双跑旧/新 Judge，分析翻转 case；不能在被评模型升级时同时偷偷换 Judge。

In [ ]:
old105=np.array([1,0,1,1,0,1,0,1]); new105=np.array([1,1,1,1,0,0,0,1]); flip_rate105=float(np.mean(old105!=new105))
manifest105={"schema":1,"judge":"model-x-snapshot","rubric":"qa-v3","pair_orders":["AB","BA"],"temperature":0,"parser":"strict-v2","human_calibration":"blind-200"}; digest105=hashlib.sha256(json.dumps(manifest105,sort_keys=True).encode()).hexdigest()
assert math.isclose(flip_rate105,.25)
assert manifest105["pair_orders"]==["AB","BA"] and manifest105["temperature"]==0
assert len(digest105)==64

## 面试总结

推荐按 **原子 rubric/schema → 已知偏差探针 → A/B swap → 风格 counterfactual → 人工盲标校准 → 重复稳定性 → 低相关多 Judge → 版本灰度** 回答。LLM Judge 是降低人工成本的测量仪器，不是把主观判断自动变成真理。

延伸阅读：[MT-Bench / LLM-as-a-Judge](https://arxiv.org/abs/2306.05685)、[Position Bias 系统研究](https://arxiv.org/abs/2406.07791)、[HELM](https://arxiv.org/abs/2211.09110)。